# 01b · Train / validation / test split

**Goal:** a split where the test score measures detection of phishing from **site owners the model has never seen**.

Code: `src/phishguard/split.py` (tests: `tests/test_split.py`). Decision: **D5** in `docs/DECISIONS.md`.

Every later notebook uses:
```python
from phishguard.split import get_splits
train_idx, val_idx, test_idx = get_splits(df)
```

In [1]:
import pandas as pd
import tldextract
from sklearn.model_selection import train_test_split
from phishguard.data import load_clean, TARGET
from phishguard.split import site_owner, get_splits

df = load_clean()
owner = site_owner(df["URL"])

## 1. Why not a plain random split?

Attackers often host many phishing URLs on one domain (`a.evil.com`, `b.evil.com`, …). If a random split puts some of them in train and some in test, the model can score well just by recognising the owner, without learning anything about phishing.

In [2]:
train, test = train_test_split(df.index, test_size=0.15, stratify=df[TARGET], random_state=42)
seen_in_train = owner[test].isin(set(owner[train]))
pd.Series({
    "all test rows": seen_in_train.mean(),
    "phishing test rows": seen_in_train[df.loc[test, TARGET] == 1].mean(),
}).rename("share whose owner also appears in train").round(3)

all test rows         0.182
phishing test rows    0.394
Name: share whose owner also appears in train, dtype: float64

About **4 in 10 phishing test URLs** come from an owner already in training. A random split would make every model look better than it is.

## 2. What counts as the same "owner"?

We compared three groupings:

In [3]:
host = df["URL"].str.extract(r"^[a-zA-Z][a-zA-Z0-9+.-]*://([^/:?#]+)", expand=False).str.lower()
icann_only = tldextract.TLDExtract(suffix_list_urls=(), include_psl_private_domains=False)
registered = host.map(lambda h: icann_only(h).top_domain_under_public_suffix or h)

def describe(groups):
    sizes = groups.value_counts()
    return {"groups": groups.nunique(),
            "phishing groups": groups[df[TARGET] == 1].nunique(),
            "rows in groups of >1": sizes[sizes > 1].sum(),
            "largest group": f"{sizes.index[0]} ({sizes.iloc[0]:,})"}

pd.DataFrame({"A. hostname": describe(host),
              "B. registered domain": describe(registered),
              "C. site owner (chosen)": describe(owner)}).T

,groups,phishing groups,rows in groups of >1,largest group
A. hostname,220078,85282,18755,"ipfs.io (1,115)"
B. registered domain,175512,43515,64363,"web.app (5,699)"
C. site owner (chosen),197712,65632,43518,"weeblysite.com (3,079)"


- **A. Hostname** leaves `a.evil.com` and `b.evil.com` in different groups, so the leak in §1 remains.
- **B. Registered domain** treats an entire hosting platform (`web.app`, `firebaseapp.com`) as one owner. Phishing collapses from 99k URLs to 44k groups, and whole platforms go missing from training. That tests "unseen hosting platforms", which is a harder and different question.
- **C. Site owner** is the registered domain, but customers of hosting platforms on the public suffix list count as separate owners (`x.web.app` ≠ `y.web.app`). We chose this.

**Limitation:** platforms that aren't on the public suffix list (e.g. `weeblysite.com`, 3,079 URLs) are still grouped as one owner.

## 3. The split

Owners are shuffled within each class and assigned to test, then validation, then train until each split holds its share of that class's rows. That keeps every owner whole and the class ratio equal across splits.

In [4]:
splits = dict(zip(["train", "val", "test"], get_splits(df)))
pd.DataFrame({name: {"rows": len(idx),
                     "share of data": round(len(idx) / len(df), 3),
                     "phishing share": round(df.loc[idx, TARGET].mean(), 4),
                     "owners": owner[idx].nunique()}
              for name, idx in splits.items()}).T.astype({"rows": int, "owners": int})

,rows,share of data,phishing share,owners
train,163944,0.70,0.4242,141423
val,35130,0.15,0.4243,28168
test,35130,0.15,0.4243,28121


In [5]:
owners = {name: set(owner[idx]) for name, idx in splits.items()}
{"train ∩ val": len(owners["train"] & owners["val"]),
 "train ∩ test": len(owners["train"] & owners["test"]),
 "val ∩ test": len(owners["val"] & owners["test"])}

{'train ∩ val': 0, 'train ∩ test': 0, 'val ∩ test': 0}

No owner appears in more than one split. Rules for everyone:
- **Tune on validation only.** Use the test set exactly once, at the end, for the final table.
- Results go through `phishguard.evaluate.evaluate(y_true, y_prob)`, so every model reports the same metrics with phishing as the positive class.